In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet("../data/pq/green/*/*")
df_yellow = spark.read.parquet("../data/pq/yellow/*/*")


In [ ]:
df_green.printSchema()

In [ ]:
df_yellow.printSchema()

In [ ]:
df_green.show(5)

In [ ]:
df_yellow.show(5)

In [ ]:
df_green = df_green.withColumnsRenamed(
    {
        "lpep_pickup_datetime": "pickup_datetime",
        "lpep_dropoff_datetime": "dropoff_datetime",
    }
)

df_yellow = df_yellow.withColumnsRenamed(
    {
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
    }
)

In [ ]:
# Get the intersection of the columns
set(df_green.columns) & set(df_yellow.columns)

In [ ]:
from pyspark.sql import functions as F

common_columns = []

for column in df_green.columns:
    if column in set(df_yellow.columns):
        common_columns.append(column)

df_green_sel = df_green \
    .select(common_columns) \
    .withColumn("type", F.lit("green"))

df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn("type", F.lit("yellow"))

In [ ]:
df_green_sel.show(5)

In [ ]:
df_yellow_sel.show(5)

In [ ]:
df_trips_data = df_green_sel.union(df_yellow_sel)

In [ ]:
df_trips_data.groupBy("type").count().show()

In [ ]:
df_trips_data.registerTempTable('trips_data')

spark.sql("""
    SELECT type, COUNT(*) AS count
    FROM trips_data
    GROUP BY type
    ORDER BY count DESC
""").show()

In [ ]:
df_result = spark.sql("""
SELECT 
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_montly_passenger_count,
    AVG(trip_distance) AS avg_montly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [ ]:
df_result.write.parquet('../data/report/revenue/')